# Module 06 — Neural networks and CNNs

> **Aquatic Guardian · Learn computer vision** — Module 06 of 09.  
> **Time:** about 2.5 hours. **Runs on:** free Google Colab, CPU is enough. **Skip if:** you have trained a CNN in PyTorch and fine-tuned a pretrained backbone.

**How to use this notebook.** In Colab choose **File → Save a copy in Drive** first, so your work is kept.
Then **Runtime → Run all**. Every cell runs without errors before you touch anything; the exercises print
feedback rather than crashing. Cells are marked:

- ▶ **Run and read** — nothing to change.
- ✏️ **Predict, then change** — write down what you expect, change one value, re-run.
- 🧩 **Your turn** — replace `None` with your code. A checker tells you whether it works. Hidden solutions follow.

### By the end you can
1. Move image data into PyTorch's `(N, C, H, W)` layout and train with mini-batches.
2. Explain what a hidden layer adds, and what a convolutional layer adds on top.
3. Train a small CNN on CPU and compare it fairly with simpler models, by condition.
4. Look inside: first-layer filters, feature maps, and a saliency map.
5. Fine-tune a pretrained backbone on real data in Colab (the upgrade path in **Mentee 1, Task 6**).

**Runtime:** about 2 minutes on Colab's free CPU. A GPU (Runtime → Change runtime type) makes it faster but isn't needed.

In [ ]:
#@title ▶ Setup: scenes and real wildlife photographs (run it; you don't need to read it yet)
import numpy as np, matplotlib.pyplot as plt, urllib.request
from pathlib import Path
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10})

PHOTO_BASE = "https://raw.githubusercontent.com/ejri/aquatic-guardian-learning-modules/main/images/"
PHOTO_FILES = {
    "reef-school": "reef-school.jpg", "manatees": "manatees.jpg", "sea-lion": "sea-lion.jpg",
    "nurse-shark": "nurse-shark.jpg", "eagle-ray": "eagle-ray.jpg", "grouper": "grouper.jpg",
    "angelfish": "angelfish.jpg", "turtle": "turtle.jpg", "noaa-honu": "noaa-honu.jpg",
    "noaa-whale": "noaa-whale.jpg", "kelp-forest": "kelp-forest.jpg", "zebra-shark": "zebra-shark.jpg",
}

def load_photo(name, max_side=480):
    """Load a real wildlife JPEG. Looks in ./images and ../images, then downloads from GitHub."""
    file = PHOTO_FILES.get(name, name if str(name).endswith(".jpg") else str(name) + ".jpg")
    arr = None
    for candidate in (Path("images") / file, Path("../images") / file, Path(file)):
        if candidate.is_file():
            arr = plt.imread(candidate)
            break
    if arr is None:
        urllib.request.urlretrieve(PHOTO_BASE + file, file)
        arr = plt.imread(file)
    if getattr(arr, "dtype", None) == np.uint8:
        arr = arr.astype(float) / 255.0
    arr = np.asarray(arr[..., :3], dtype=float)
    h, w = arr.shape[:2]
    scale = max_side / max(h, w)
    if scale < 1:
        yy = (np.linspace(0, h - 1, max(1, int(h * scale)))).astype(int)
        xx = (np.linspace(0, w - 1, max(1, int(w * scale)))).astype(int)
        arr = arr[yy][:, xx]
    return arr


# Illustrative attenuation coefficients beta (1/m) for R, G, B, and the colour of the veiling light.
# These are round teaching numbers with the right ordering, not measurements of any real site.
WATER = {
    "clear_ocean":    dict(beta=(0.40, 0.07, 0.04), veil=(0.02, 0.20, 0.42)),
    "coastal":        dict(beta=(0.55, 0.20, 0.30), veil=(0.05, 0.32, 0.27)),
    "turbid_harbour": dict(beta=(0.95, 0.60, 0.75), veil=(0.14, 0.30, 0.22)),
}

def _fish_mask(H, W, cx, cy, L, ang, aspect=4.0):
    """Ellipse body plus a triangular tail. L is total length (snout to tail tip) in pixels."""
    yy, xx = np.mgrid[0:H, 0:W]
    a = L / 2.25; b = a / aspect
    X = (xx - cx) * np.cos(ang) + (yy - cy) * np.sin(ang)
    Y = -(xx - cx) * np.sin(ang) + (yy - cy) * np.cos(ang)
    body = (X / a) ** 2 + (Y / b) ** 2 <= 1
    tx = -X - 0.80 * a
    tail = (tx >= 0) & (tx <= 0.45 * a) & (np.abs(Y) <= 1.3 * b * tx / (0.45 * a))
    return body | tail, X, Y, a, b

FISH_COLOURS = [(0.97, 0.55, 0.12), (0.96, 0.84, 0.22), (0.80, 0.82, 0.84), (0.93, 0.35, 0.25)]

def make_scene(n_fish=4, H=120, W=180, seed=0, bg_seed=None, fish_dist=(2.0, 6.0), bg_dist=12.0,
               length_px=(22, 40), min_gap=0.55, touching=False, aspect=(4.0, 4.0)):
    """Returns a dict with the true scene radiance J (what you'd see in air), a distance map in metres,
    one boolean mask and one box (x1, y1, x2, y2) per fish, and per-fish truth."""
    r = np.random.default_rng(seed)
    rb = np.random.default_rng(seed if bg_seed is None else bg_seed)
    yy, xx = np.mgrid[0:H, 0:W] / max(H, W)
    J = np.stack([0.48 + 0.10 * yy, 0.43 + 0.08 * yy, 0.34 + 0.04 * xx], -1)       # reef wall / sand
    J = J + 0.04 * rb.standard_normal((H, W, 1)) + 0.015 * rb.standard_normal((H, W, 3))
    dist = np.full((H, W), float(bg_dist))
    masks, boxes, truth, placed = [], [], [], []
    for i in range(n_fish):
        for _ in range(200):
            L = r.uniform(*length_px); ang = r.uniform(-0.6, 0.6) + (np.pi if r.random() < 0.5 else 0)
            cx = r.uniform(0.15 * W, 0.85 * W); cy = r.uniform(0.18 * H, 0.82 * H)
            gap = 0.25 if touching else min_gap
            if all((cx - px) ** 2 + (cy - py) ** 2 > (gap * (L + pl)) ** 2 for px, py, pl in placed):
                break
        placed.append((cx, cy, L))
        asp = aspect[0] if aspect[0] == aspect[1] else r.uniform(*aspect)
        m, X, Y, a, b = _fish_mask(H, W, cx, cy, L, ang, asp)
        col = np.array(FISH_COLOURS[i % len(FISH_COLOURS)])
        J[m] = col
        stripe = m & (np.abs(X - 0.15 * a) < 0.09 * a)
        J[stripe] = (0.95, 0.95, 0.92)
        eye = (X - 0.62 * a) ** 2 + (Y - 0.15 * b) ** 2 < (0.12 * b + 0.6) ** 2
        J[eye & m] = (0.05, 0.05, 0.08)
        d = r.uniform(*fish_dist); dist[m] = d
        ys, xs = np.where(m)
        masks.append(m); boxes.append((int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1))
        truth.append(dict(cx=float(cx), cy=float(cy), length_px=float(L), angle=float(ang), dist_m=float(d), aspect=float(asp)))
    return dict(J=np.clip(J, 0, 1), dist=dist, masks=masks, boxes=boxes, truth=truth)

def underwater(J, dist, water="coastal", light=1.0, photons=None, seed=0):
    """Simple image-formation model, per colour channel c:
         I_c = light * ( J_c * t_c  +  B_c * (1 - t_c) ),   t_c = exp(-beta_c * distance)
    photons: if set, adds shot noise (fewer photons -> noisier). Lower `light` means fewer photons."""
    beta = np.array(WATER[water]["beta"]); B = np.array(WATER[water]["veil"])
    t = np.exp(-beta[None, None, :] * dist[..., None])
    I = light * (J * t + B * (1 - t))
    if photons:
        rng = np.random.default_rng(seed)
        n = photons * light
        I = rng.poisson(np.clip(I, 0, None) / max(light, 1e-6) * n) / n * light
    return np.clip(I, 0, 1)

def degrade(img, turbidity=0.0, low_light=0.0, blur_k=1):
    """The degradation utility used in the Aquatic Guardian project notebooks (copied verbatim in spirit)."""
    out = img.copy()
    if blur_k > 1:
        pad = blur_k // 2; P = np.pad(out, ((pad, pad), (pad, pad), (0, 0)), mode="edge"); acc = np.zeros_like(out)
        for dy in range(blur_k):
            for dx in range(blur_k): acc += P[dy:dy + out.shape[0], dx:dx + out.shape[1], :]
        out = acc / (blur_k * blur_k)
    if turbidity > 0:
        veil = np.array([0.20, 0.42, 0.45]); out = (1 - turbidity) * out + turbidity * veil
        out = (1 - 0.5 * turbidity) * out + 0.5 * turbidity * out.mean()
    if low_light > 0: out = (out * (1 - 0.85 * low_light)) ** (1 + 0.6 * low_light)
    return np.clip(out, 0, 1)

def show(images, titles=None, size=3.2, cmap=None):
    images = images if isinstance(images, (list, tuple)) else [images]
    fig, ax = plt.subplots(1, len(images), figsize=(size * len(images), size * 0.72))
    ax = np.atleast_1d(ax)
    for i, (a, im) in enumerate(zip(ax, images)):
        a.imshow(im, cmap=cmap if im.ndim == 2 else None, vmin=0 if im.ndim == 2 and cmap else None,
                 vmax=1 if im.ndim == 2 and cmap else None)
        a.set_title(titles[i] if titles else ""); a.axis("off")
    plt.tight_layout(); plt.show()

def draw_boxes(img, boxes, colour=(1.0, 0.85, 0.0)):
    out = img.copy()
    for (x1, y1, x2, y2) in boxes:
        x2 -= 1; y2 -= 1
        out[y1, x1:x2 + 1] = colour; out[y2, x1:x2 + 1] = colour
        out[y1:y2 + 1, x1] = colour; out[y1:y2 + 1, x2] = colour
    return out
print("Setup ready. Water types:", list(WATER))

In [ ]:
#@title ▶ Setup: a staged dataset of short camera clips (fish / non_fish / empty_frame)
CLASSES = ["fish", "non_fish", "empty_frame"]     # Mentee 1's Level-1 classes
HW = 48                                           # frame size in pixels
HABITATS = {   # background colour in air, water type, typical object distance (m)
    "reef":    dict(bg=(0.52, 0.47, 0.36), water="clear_ocean",    dist=(1.0, 4.0)),
    "kelp":    dict(bg=(0.30, 0.34, 0.22), water="coastal",        dist=(1.0, 3.5)),
    "harbour": dict(bg=(0.36, 0.33, 0.28), water="turbid_harbour", dist=(0.6, 2.2)),
}

def new_session(cls, habitat, r):
    h = HABITATS[habitat]
    return dict(cls=cls, habitat=habitat, water=h["water"], bg=np.array(h["bg"]) + r.normal(0, 0.03, 3),
                obj_dist=r.uniform(*h["dist"]), bg_dist=r.uniform(6, 12), photons=r.uniform(250, 900),
                x0=r.uniform(14, 34), y0=r.uniform(14, 34), vx=r.normal(0, 0.6), vy=r.normal(0, 0.3),
                L=r.uniform(16, 30), ang=r.uniform(0, np.pi), aspect=r.uniform(3.2, 4.5),
                col=np.array(FISH_COLOURS[r.integers(len(FISH_COLOURS))]) + r.normal(0, 0.05, 3),
                blob_col=r.uniform(0.25, 0.6, 3), blob_r=r.uniform(4, 8), blob_aspect=r.uniform(1.0, 1.5))

def render_frame(s, i, r):
    yy, xx = np.mgrid[0:HW, 0:HW] / HW
    J = s["bg"] + 0.06 * np.stack([yy, 0.8 * yy, 0.5 * xx], -1) + 0.03 * r.standard_normal((HW, HW, 1))
    dist = np.full((HW, HW), s["bg_dist"])
    cx = np.clip(s["x0"] + s["vx"] * i, 8, HW - 8); cy = np.clip(s["y0"] + s["vy"] * i, 8, HW - 8)
    if s["cls"] == "fish":
        m = _fish_mask(HW, HW, cx, cy, s["L"], s["ang"] + 0.02 * i, s["aspect"])[0]; J[m] = s["col"]; dist[m] = s["obj_dist"]
    elif s["cls"] == "non_fish":     # crab, rock, jellyfish: compact blobs
        m = _fish_mask(HW, HW, cx, cy, 2.25 * s["blob_r"], s["ang"], s["blob_aspect"])[0]
        m &= ((np.mgrid[0:HW, 0:HW][1] - cx) ** 2 + (np.mgrid[0:HW, 0:HW][0] - cy) ** 2) <= (1.2 * s["blob_r"]) ** 2
        J[m] = s["blob_col"]; dist[m] = s["obj_dist"]
    return underwater(np.clip(J, 0, 1), dist, s["water"], photons=s["photons"], seed=int(r.integers(1 << 30)))

def make_clips(n_sessions_per_class=40, frames_per_clip=12, habitats=("reef", "kelp", "harbour"), seed=0):
    """Each session is one short clip: one object (or none) drifting past a fixed camera.
    Returns images X (N, HW, HW, 3), labels y (N,), session ids (N,), habitat names (N,)."""
    r = np.random.default_rng(seed); X, y, sess, hab = [], [], [], []
    sid = 0
    for ci, cls in enumerate(CLASSES):
        for k in range(n_sessions_per_class):
            habitat = habitats[k % len(habitats)]
            s = new_session(cls, habitat, r)
            for i in range(frames_per_clip):
                X.append(render_frame(s, i, r)); y.append(ci); sess.append(sid); hab.append(habitat)
            sid += 1
    return np.array(X), np.array(y), np.array(sess), np.array(hab)
print("Clip generator ready. Classes:", CLASSES)

In [ ]:
import time, pandas as pd
try:
    import torch, torch.nn as nn, torch.nn.functional as Fnn
    HAVE_TORCH = True
except ImportError:
    HAVE_TORCH = False
    print("PyTorch is not installed here. In Colab it is preinstalled; locally: pip install torch torchvision")
if HAVE_TORCH:
    torch.manual_seed(0)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("PyTorch", torch.__version__, "| device:", device)

## 1. Data in PyTorch layout ▶

NumPy images are `(N, H, W, C)`. PyTorch convolutions expect `(N, C, H, W)`: channels first.

In [ ]:
Xtr, ytr, _, htr = make_clips(n_sessions_per_class=45, seed=10)
Xte, yte, _, hte = make_clips(n_sessions_per_class=15, seed=99)
HABITATS["harbour_far"] = dict(HABITATS["harbour"], dist=(2.5, 4.0)); HABITATS["reef_far"] = dict(HABITATS["reef"], dist=(5.0, 8.0))
Xfar, yfar, _, _ = make_clips(n_sessions_per_class=15, habitats=("reef_far", "harbour_far"), seed=7)
print("numpy:", Xtr.shape)
if HAVE_TORCH:
    def to_t(X, norm=False):
        X = np.asarray(X, dtype=np.float32)
        if norm:   # per-image standardisation, explained in section 3
            X = (X - X.mean((1, 2), keepdims=True)) / (X.std((1, 2), keepdims=True) + 1e-3)
        return torch.tensor(np.transpose(X, (0, 3, 1, 2)))
    Ttr, Tte = to_t(Xtr), to_t(Xte); Ytr, Yte = torch.tensor(ytr), torch.tensor(yte)
    print("torch:", tuple(Ttr.shape), "(N, C, H, W)")

## 2. Three models of increasing structure ▶

1. **Linear** on raw pixels: one weight per pixel per class. Same maths as Module 04's softmax, but no hand-made features.
2. **MLP**: one hidden layer with a nonlinearity (ReLU). It can learn combinations of pixels.
3. **CNN**: small filters slid across the image (Module 03's convolution, but the kernel values are *learned*),
   then pooling to shrink the image. It shares weights across positions, so a fish in the corner and a fish in
   the centre are detected by the same filters.

In [ ]:
if HAVE_TORCH:
    class Linear(nn.Module):
        def __init__(s): super().__init__(); s.f = nn.Linear(3 * HW * HW, 3)
        def forward(s, x): return s.f(x.flatten(1))
    class MLP(nn.Module):
        def __init__(s, h=128): super().__init__(); s.f = nn.Sequential(nn.Flatten(), nn.Linear(3 * HW * HW, h), nn.ReLU(), nn.Linear(h, 3))
        def forward(s, x): return s.f(x)
    class SmallCNN(nn.Module):
        def __init__(s, w=16):
            super().__init__()
            s.features = nn.Sequential(
                nn.Conv2d(3, w, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),          # 48 -> 24
                nn.Conv2d(w, 2 * w, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),      # 24 -> 12
                nn.Conv2d(2 * w, 4 * w, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))  # 12 -> 6
            s.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(4 * w, 3))
        def forward(s, x): return s.head(s.features(x))

    def train(model, T, Y, epochs=8, bs=64, lr=2e-3, seed=0):
        torch.manual_seed(seed); model = model.to(device); opt = torch.optim.Adam(model.parameters(), lr)
        hist = []
        for ep in range(epochs):
            model.train(); perm = torch.randperm(len(T)); tot = 0.0
            for i in range(0, len(T), bs):
                idx = perm[i:i + bs]; xb, yb = T[idx].to(device), Y[idx].to(device)
                opt.zero_grad(); loss = Fnn.cross_entropy(model(xb), yb); loss.backward(); opt.step(); tot += loss.item() * len(idx)
            hist.append(tot / len(T))
        return model, hist

    @torch.no_grad()
    def accuracy(model, X, y):
        model.eval(); p = model(to_t(X, getattr(model, "norm", False)).to(device)).argmax(1).cpu().numpy(); return (p == y).mean()

    def by_condition(model):
        return {"clean": accuracy(model, Xte, yte),
                "turbid 0.3": accuracy(model, np.array([degrade(im, turbidity=0.3) for im in Xte]), yte),
                "low light 0.5": accuracy(model, np.array([degrade(im, low_light=0.5) for im in Xte]), yte),
                "blur k=3": accuracy(model, np.array([degrade(im, blur_k=3) for im in Xte]), yte),
                "farther (physics)": accuracy(model, Xfar, yfar)}

    results, curves, models = {}, {}, {}
    for name, M_ in [("linear", Linear()), ("MLP", MLP()), ("small CNN", SmallCNN())]:
        t0 = time.time(); m, h = train(M_, Ttr, Ytr)
        models[name], curves[name] = m, h
        results[name] = by_condition(m); results[name]["train s"] = round(time.time() - t0, 1)
        results[name]["params"] = sum(p.numel() for p in m.parameters())
    plt.figure(figsize=(5, 2.6))
    for k, h in curves.items(): plt.plot(h, label=k)
    plt.xlabel("epoch"); plt.ylabel("training loss"); plt.legend(); plt.show()
    display(pd.DataFrame(results).T.round(3))

✏️ **Interpret:** the MLP has roughly 40 times more parameters than the CNN, yet the CNN does better, because
its filters are shared across positions and suit images. Do any of the three beat Module 05's six hand-made
features (about 0.93 on clean frames)? If not, why might that be? (Hint: look at how dim and low-contrast the raw
frames are, and how much they differ in colour from one habitat to the next.)

## 3. Preprocessing: one line that changes a lot ▶

Our frames differ mostly in overall **colour cast and brightness** from habitat to habitat. Subtracting each
frame's own mean and dividing by its own standard deviation (per channel) removes that, so the network can
spend its capacity on shape. Same CNN, same epochs, one extra line.

In [ ]:
if HAVE_TORCH:
    t0 = time.time(); cnn_n, _ = train(SmallCNN(), to_t(Xtr, norm=True), Ytr); cnn_n.norm = True
    models["small CNN, normalised"] = cnn_n
    results["small CNN, normalised"] = by_condition(cnn_n)
    results["small CNN, normalised"]["train s"] = round(time.time() - t0, 1)
    results["small CNN, normalised"]["params"] = results["small CNN"]["params"]
    display(pd.DataFrame(results).T.round(3))

✏️ **Interpret, carefully.** Look at which columns improve and which don't. Then look at `degrade()` in the
setup cell: its turbidity step is (almost) `a * image + constant` for the whole frame. Per-image standardisation
cancels exactly that kind of change. Blur changes the *spatial* structure, which standardisation can't undo.

This is a result about the **test**, not only about the model: a robustness score measured with `degrade()` can
look excellent for a model that simply normalises away `degrade()`'s particular distortion. Compare the improvement
on the physically farther objects. **Always test with more than one degradation model.**

## 4. Look inside the CNN ▶

**First-layer filters** are 3×3 colour kernels. **Feature maps** show where each filter responds.

In [ ]:
if HAVE_TORCH:
    cnn = models["small CNN, normalised"].cpu()
    w = cnn.features[0].weight.detach().numpy()            # (16, 3, 3, 3)
    fig, ax = plt.subplots(2, 8, figsize=(10, 2.8))
    for i, a in enumerate(ax.ravel()):
        k = np.transpose(w[i], (1, 2, 0)); k = (k - k.min()) / (k.max() - k.min() + 1e-9)
        a.imshow(k, interpolation="nearest"); a.axis("off")
    fig.suptitle("16 learned first-layer filters (each 3x3, colour)", fontsize=9); plt.show()
    fish_frame = Xte[np.where(yte == 0)[0][5]]
    with torch.no_grad(): fmap = cnn.features[:2](to_t(fish_frame[None], norm=True))[0].numpy()
    fig, ax = plt.subplots(1, 7, figsize=(13, 2.1)); ax[0].imshow(fish_frame); ax[0].set_title("input"); ax[0].axis("off")
    for i, a in enumerate(ax[1:]): a.imshow(fmap[i], cmap="magma"); a.set_title(f"filter {i}"); a.axis("off")
    plt.show(); cnn.to(device)

## 5. What is the model looking at? A saliency map ▶

Take the gradient of the predicted class score with respect to the input pixels. Large values mean "changing this
pixel would change the decision most". If the bright region isn't on the animal, the model may be using a shortcut
(a background colour that happens to correlate with the class in your data).

In [ ]:
if HAVE_TORCH:
    cnn = models["small CNN, normalised"].cpu().eval()
    picks = [np.where(yte == c)[0][7] for c in range(3)]
    fig, ax = plt.subplots(2, 3, figsize=(7.5, 5))
    for j, k in enumerate(picks):
        x = to_t(Xte[k][None], norm=True).requires_grad_(True); out = cnn(x); c = out.argmax(1).item()
        out[0, c].backward(); sal = x.grad.abs()[0].max(0).values.numpy()
        ax[0, j].imshow(Xte[k]); ax[0, j].set_title(f"true {CLASSES[yte[k]]}\npred {CLASSES[c]}", fontsize=8); ax[0, j].axis("off")
        ax[1, j].imshow(sal, cmap="inferno"); ax[1, j].axis("off")
    plt.tight_layout(); plt.show(); cnn.to(device)

## 6. Augmentation on top ▶

In [ ]:
if HAVE_TORCH:
    r = np.random.default_rng(3); aug = []
    for im in Xtr:
        kw = [dict(turbidity=0.35), dict(low_light=0.5), dict(turbidity=0.25, blur_k=3), dict(blur_k=3)][r.integers(4)]
        aug.append(degrade(im, **kw))
    Taug = torch.cat([to_t(Xtr, norm=True), to_t(np.array(aug), norm=True)]); Yaug = torch.cat([Ytr, Ytr])
    t0 = time.time(); cnn_aug, _ = train(SmallCNN(), Taug, Yaug); cnn_aug.norm = True
    key = "small CNN, normalised + augmentation"
    results[key] = by_condition(cnn_aug); results[key]["train s"] = round(time.time() - t0, 1)
    results[key]["params"] = results["small CNN"]["params"]
    display(pd.DataFrame(results).T.round(3))

✏️ **Interpret:** compare the four `degrade()` columns with the *farther (physics)* column. A score that looks
almost perfect under one degradation model can hide a large gap under another. Write both into any report, and
treat real turbid footage as the only test that settles it.

## 7. Size and latency ▶

The project's later phase runs models on a small device, so report parameters and time per image, measured the same
way for every model (Mentee 1, Task 9).

In [ ]:
if HAVE_TORCH:
    x1 = to_t(Xte[:1]).to(device)
    for name, m in [("linear", models["linear"]), ("MLP", models["MLP"]), ("small CNN", models["small CNN, normalised"])]:
        m.eval()
        with torch.no_grad():
            for _ in range(5): m(x1)
            t0 = time.time()
            for _ in range(100): m(x1)
        print(f"{name:10s} {sum(p.numel() for p in m.parameters()):>8,d} params   {(time.time() - t0) * 10:.2f} ms/image on {device}")

## 8. Real data: fine-tuning a pretrained backbone (Colab)

A network pretrained on millions of everyday photos already knows edges, textures and shapes. Replacing its last
layer and training briefly on your data is **transfer learning**. This is the Mentee 1 Task 6 upgrade path.

Set `RUN_REAL_DATA = True` in Colab, arrange images as `data_dir/<class>/*.jpg` (for example, frames exported from
DeepFish or Brackish, mapped onto fish / non_fish / empty_frame, **split by clip**), and record the licence.

In [ ]:
RUN_REAL_DATA = False
DATA_DIR = "/content/aquatic_frames"     # data_dir/fish, data_dir/non_fish, data_dir/empty_frame

def finetune_mobilenet(data_dir, epochs=5, img=128, batch=32):
    from torchvision import datasets, transforms, models as tvm
    from torch.utils.data import DataLoader
    tf = transforms.Compose([transforms.Resize((img, img)), transforms.ToTensor(),
                             transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    tr = datasets.ImageFolder(f"{data_dir}/train", tf); va = datasets.ImageFolder(f"{data_dir}/val", tf)   # split by clip beforehand
    net = tvm.mobilenet_v3_small(weights="DEFAULT")
    net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, len(tr.classes))
    for p in net.features.parameters(): p.requires_grad = False          # train the new head first
    net = net.to(device); opt = torch.optim.Adam([p for p in net.parameters() if p.requires_grad], 1e-3)
    for ep in range(epochs):
        net.train()
        for xb, yb in DataLoader(tr, batch, shuffle=True):
            opt.zero_grad(); Fnn.cross_entropy(net(xb.to(device)), yb.to(device)).backward(); opt.step()
    net.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in DataLoader(va, batch):
            correct += (net(xb.to(device)).argmax(1).cpu() == yb).sum().item(); total += len(yb)
    print(f"validation accuracy: {correct / total:.3f}  classes: {tr.classes}")
    return net

if HAVE_TORCH and RUN_REAL_DATA:
    real_model = finetune_mobilenet(DATA_DIR)
else:
    print("Skipped (RUN_REAL_DATA = False). In Colab: prepare DATA_DIR/train and DATA_DIR/val, then set it to True.")

### 🧩 Exercise — shapes through the network

Our CNN takes a 48×48 image. Each `Conv2d(..., 3, padding=1)` keeps the size; each `MaxPool2d(2)` halves it.
What is the spatial size (height = width) of the output of `cnn.features`? Work it out, then store it in
`spatial_size`. A checker runs a real tensor through to confirm.

In [ ]:
spatial_size = None   # YOUR ANSWER (an integer)

def check_size(v):
    if v is None: print("Not yet: replace None with an integer."); return
    if HAVE_TORCH:
        with torch.no_grad(): s = models["small CNN"].cpu().features(torch.zeros(1, 3, HW, HW)).shape[-1]
        models["small CNN"].to(device)
    else: s = HW // 8
    print("✅ Correct:" if v == s else "❌ Not quite: three halvings of 48.", f"actual size = {s}")
check_size(spatial_size)

In [ ]:
#@title Solution (double-click to show)
spatial_size = 48 // 2 // 2 // 2
check_size(spatial_size)

## Check yourself

<details><summary>1. Why does the CNN need far fewer parameters than a linear model on raw pixels?</summary>Its small filters are reused at every position (weight sharing), instead of one weight per pixel per class.</details>
<details><summary>2. Your saliency map lights up the background, not the fish. What might be wrong?</summary>A shortcut: some background property correlates with the label in your training data. Check class balance per habitat and per clip.</details>
<details><summary>3. When fine-tuning a pretrained network, why freeze the early layers first?</summary>They hold general features (edges, textures). Training only the new head first is faster and less likely to overfit a small dataset.</details>
<details><summary>4. Your model scores 0.97 under degrade(turbidity=0.3). Is it robust to turbid water?</summary>Not necessarily. It may only have learned to cancel that function's particular distortion. Test with a different degradation model (for example the physics model) and, ideally, real turbid footage.</details>

**Next:** Module 07, *Detection, measurement and calibration* — from "is there a fish?" to "where, how big, and how heavy?".